# CSV 排错：从“能运行”到“结果可核对”

这是一个自制数据练习，不是实验结论。目标：处理两种明确的缺测标记，保留排除记录，再画出观测值和均值。无需 AI 账号或付费订阅；可先独立运行，再用文末模板向任意 AI 工具提问。

In [1]:
import sys
import pandas as pd
import matplotlib
print("Python", sys.version.split()[0], "pandas", pd.__version__, "matplotlib", matplotlib.__version__)

Python 3.9.6 pandas 2.3.3 matplotlib 3.9.4


In [2]:
# 错误示例（不执行）：signal 包含 not_measured，不能直接转成浮点数。
# df = pd.read_csv("example.csv")
# df["signal"] = df["signal"].astype(float)
# ValueError: could not convert string to float: 'not_measured'
# 不要直接填 0，也不要对整张表无差别 dropna()。

## 完整修复
先检查输入，再处理明确的缺测，最后保存核验表和图。

In [3]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

source = Path("example.csv")
if not source.is_file():
    raise FileNotFoundError("Open the notebook from the folder containing example.csv.")
raw = pd.read_csv(source, dtype=str, keep_default_na=False)
required = {"sample_id", "group", "signal"}
if not required.issubset(raw.columns):
    raise ValueError("CSV must contain sample_id, group and signal columns.")
if raw["sample_id"].eq("").any() or raw["sample_id"].duplicated().any():
    raise ValueError("Sample IDs must be present and unique.")
if not raw["group"].isin(["control", "treatment"]).all():
    raise ValueError("Unknown or missing group: check the input before analysis.")
values = pd.to_numeric(raw["signal"], errors="coerce")
# Only the two documented missing markers are accepted; stop on unexpected text.
known_missing = raw["signal"].isin(["", "not_measured"])
if (values.isna() & ~known_missing).any():
    raise ValueError("Unexpected non-numeric signal: check the source data.")
if values.isin([float("inf"), -float("inf")]).any():
    raise ValueError("Signal must be finite.")
clean = raw.assign(signal=values)
excluded = clean.loc[known_missing, ["sample_id", "group"]].copy()
excluded["reason"] = "signal_missing"
valid = clean.loc[~known_missing].copy()
summary = valid.groupby("group")["signal"].agg(n="count", mean="mean").reindex(["control", "treatment"])
if summary["n"].isna().any() or summary["n"].eq(0).any():
    raise ValueError("Each group needs at least one valid observation.")
assert len(raw) == len(valid) + len(excluded)
print(f"Input={len(raw)}; included={len(valid)}; excluded={len(excluded)}")
print(summary.to_string())
print("Excluded rows:\n" + excluded.to_string(index=False))
summary.to_csv("summary.csv")
excluded.to_csv("excluded.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for x, group in enumerate(summary.index):
    ys = valid.loc[valid["group"].eq(group), "signal"].to_numpy()
    offsets = [(i - (len(ys)-1)/2)*0.08 for i in range(len(ys))]
    ax.scatter([x+v for v in offsets], ys, color="#2563eb", label="Observed" if x == 0 else None)
    ax.hlines(summary.loc[group, "mean"], x-.2, x+.2, colors="#0f172a", label="Mean" if x == 0 else None)
ax.set_xticks([0, 1], [f"{g} (n={int(summary.loc[g, 'n'])})" for g in summary.index])
ax.set_ylabel("Signal (arbitrary units)")
ax.set_title("Synthetic data: observations and means")
ax.legend(frameon=False)
fig.savefig("result.png", dpi=160)
plt.close(fig)


Input=8; included=6; excluded=2
           n  mean
group             
control    3  11.0
treatment  3  16.0
Excluded rows:
sample_id     group         reason
       C4   control signal_missing
       T4 treatment signal_missing


## 核验答案

输入 8 行；纳入 6 行；排除 C4、T4 共 2 行。control 有效 n=3、均值 11；treatment 有效 n=3、均值 16。原始 CSV 不应被改写。

这里的缺测处理规则仅用于练习，不能直接替代真实研究的预设分析方案。均值差不是统计显著性或因果结论。

## 可复制的提问模板

我用 Python / pandas 读取 CSV，signal 列出现空值和 not_measured。报错为：ValueError: could not convert string to float: 'not_measured'。

任务：按组画出每个观测值和均值，报告每组有效样本数。仅将空字符串和 not_measured 作为缺测，保留排除行；其他异常应报错。不要填 0、推断缺失值、添加显著性结论或无差别删除整行。

环境：请粘贴下一单元格显示的实际版本。
最小复现数据：只粘贴本包自制的 example.csv。
相关代码：请粘贴触发错误的最小片段。

请解释错误原因，给出可运行修复，并说明如何独立核验样本数、均值及排除记录。真实项目先确认数据可向外部工具提供。